# Notebook 01 - Consolidacion de datos fiscales

**Fuente:** Secretaria de Hacienda - https://www.argentina.gob.ar/economia/sechacienda/infoestadistica  
**Datos:** Sector Publico Base Caja (AIF) + Informe Mensual de Ingresos y Gastos (IMIG)  
**Cobertura:** enero 2020 - abril 2026

Este notebook lee los datasets ya consolidados desde GitHub y los exporta como un unico archivo Excel con multiples hojas, listo para usar en el **Notebook 02 de analisis**.

> **Nota:** La consolidacion desde los Excel originales se hace localmente con `python src/consolidate.py`. Los CSVs resultantes ya estan publicados en este repositorio.

In [ ]:
# Celda 1 - Instalacion de dependencias
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas openpyxl

import pandas as pd
import openpyxl
import os
print('OK - dependencias cargadas')

In [ ]:
# Celda 2 - URLs de los datos (rama main)
REPO_RAW = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'

AIF_URL  = f'{REPO_RAW}/output/aif_consolidado.csv'
IMIG_URL = f'{REPO_RAW}/output/imig_consolidado.csv'

print('Descargando datasets...')
df_aif  = pd.read_csv(AIF_URL,  parse_dates=['fecha'])
df_imig = pd.read_csv(IMIG_URL, parse_dates=['fecha'])

print(f'AIF : {len(df_aif):,} registros | {df_aif["fecha"].min().strftime("%Y-%m")} a {df_aif["fecha"].max().strftime("%Y-%m")}')
print(f'IMIG: {len(df_imig):,} registros')
df_aif.head(3)

In [ ]:
# Celda 3 - Resumen de cobertura
print('=== AIF - Conceptos disponibles ===')
print(df_aif['concepto_codigo'].value_counts().to_string())
print()
print('=== AIF - Subsectores ===')
print(df_aif['subsector'].value_counts().to_string())
print()
print('=== AIF - Periodos ===')
print(df_aif['periodo'].value_counts().to_string())
print()
print('=== Meses cubiertos por año ===')
print(df_aif[df_aif['periodo']=='mensual'].groupby('anio')['mes'].nunique().to_string())

In [ ]:
# Celda 4 - Exportar a Excel consolidado (un archivo, multiples hojas)
OUTPUT_FILE = 'datos_fiscales_consolidado.xlsx'

print(f'Exportando a {OUTPUT_FILE}...')

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:

    # Hoja 1: AIF mensual - todos los subsectores
    df_aif[df_aif['periodo'] == 'mensual'].to_excel(
        writer, sheet_name='AIF_mensual', index=False)

    # Hoja 2: AIF acumulado
    df_aif[df_aif['periodo'] == 'acumulado'].to_excel(
        writer, sheet_name='AIF_acumulado', index=False)

    # Hoja 3: Resultado primario y financiero (pivot por fecha)
    df_resultados = df_aif[
        (df_aif['periodo'] == 'mensual') &
        (df_aif['subsector'] == 'total_adm_nacional') &
        (df_aif['concepto_codigo'].isin([
            'I_INGRESOS_CORRIENTES',
            'II_GASTOS_CORRIENTES',
            'XIV_RESULTADO_PRIMARIO',
            'XV_RESULTADO_FINANCIERO',
            'II2_INTERESES'
        ]))
    ].pivot_table(
        index='fecha',
        columns='concepto_codigo',
        values='valor_millones_pesos'
    ).reset_index()
    df_resultados.to_excel(writer, sheet_name='Resultado_pivot', index=False)

    # Hoja 4: Transferencias a provincias
    df_prov = df_aif[
        (df_aif['periodo'] == 'mensual') &
        (df_aif['concepto_codigo'].isin([
            'II4b1_TRANSF_PROVINCIAS_CABA',
            'V2a_TRANSF_CAPITAL_PROVINCIAS'
        ]))
    ]
    df_prov.to_excel(writer, sheet_name='Transferencias_provincias', index=False)

    # Hoja 5: IMIG
    df_imig.to_excel(writer, sheet_name='IMIG', index=False)

print(f'Archivo generado: {OUTPUT_FILE}')
print(f'Hojas: AIF_mensual | AIF_acumulado | Resultado_pivot | Transferencias_provincias | IMIG')

In [ ]:
# Celda 5 - Descargar el Excel (solo en Colab)
if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Descarga iniciada')
else:
    print(f'Archivo guardado localmente en: {os.path.abspath(OUTPUT_FILE)}')